# Section VI-D — Neuron-Level Causal Validation of Modality Competition (v3)

**Restart rationale:** v1 and v2 evaluated the correct, SHA-verified RML checkpoints against a fabricated `meta.pkl` that does not exist in the source data archive. All prior Phase A-D numbers are retracted. See `docs/adr/0004-v3-restart-data-provenance-and-fidelity-gate.md` and `journals/2026-08-09.md` for the full diagnosis.

**What is different in v3:**
- Data loads from the canonical `BIG_DATA_RAW_PROCESSED_FACE` + `meta_one_hot_label_six_categories.pkl` source (the same one `main.py` used to train and evaluate every dataset in the thesis), filtered to RML split IDs — not the retired `RML_RAW_PROCESSED_Face` folder.
- Day 0 ends in a **halting** fidelity gate: `assert` that base/fine-tuned test accuracy reproduce the paper's published RML numbers (76.39% / 79.86%, Figure 4.3) within 1% absolute, on the canonical 144-sample test split. If the gate fails, **stop** — do not run Week 1 onward on unverified data.
- Label ordering is the fixed constant from `getEmotionDict()` (`{'ang':0,'dis':1,'fea':2,'hap':3,'sad':4,'sur':5}`). There is no label-permutation solver. If accuracy doesn't match, that is a stop signal, not something to search around.
- Checkpoints load with `strict=True`.
- Before any real ablation sweep, a falsification pair runs: full 64-dim audio knockout (must move accuracy substantially) and a 5-random-dim control (must be near zero). This is cheap insurance against a repeat of the v1/v2 failure mode.

**Run order:** execute cells top to bottom on a Colab GPU runtime (T4/V100). Do not skip the Day 0 fidelity gate cell.

# Day 0 — Environment & Path Setup

In [ ]:
# -- Global Environment & Path Setup --
import os
import sys
import random
import hashlib
import types
import builtins
import math
import numpy as np
import torch
import torch.nn as nn

builtins.torch = torch
builtins.nn = nn

# Colab Python 3.12 / PyTorch 2.2.2 compatibility shims.
# These only patch missing attributes so newer transformers/torch code paths
# don't crash on Colab's pinned older torch build; they do not change any
# numerical behavior of the model itself.
if not hasattr(torch, 'get_default_device'):
    torch.get_default_device = lambda: torch.device("cpu")
if not hasattr(torch, 'set_default_device'):
    torch.set_default_device = lambda dev: None
if not hasattr(torch, 'is_compiling'):
    torch.is_compiling = lambda: False
if not hasattr(torch, 'compiler'):
    comp_mod = types.ModuleType('compiler')
    comp_mod.is_compiling = lambda: False
    torch.compiler = comp_mod

import transformers
import transformers.utils.import_utils as import_utils
import_utils.BACKENDS_MAPPING["torch"] = (lambda: True, "PyTorch library")
import_utils.is_torch_available = lambda: True
import_utils._torch_available = True
if hasattr(transformers, 'is_torch_available'):
    transformers.is_torch_available = lambda: True
try:
    from transformers.models.albert.modeling_albert import AlbertModel as RealAlbertModel
    transformers.AlbertModel = RealAlbertModel
    sys.modules['transformers'].AlbertModel = RealAlbertModel
except Exception as e:
    print(f"Notice on AlbertModel binding: {e}")

try:
    import facenet_pytorch  # noqa: F401
except ImportError:
    os.system(f"{sys.executable} -m pip install -q facenet-pytorch")

def set_deterministic_seed(seed: int = 0) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_deterministic_seed(0)

# -- Path resolution & Google Drive mount --
project_path = None
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

for candidate in ['/content/drive/MyDrive/multimodal-causal-ablation',
                  '/content/multimodal-causal-ablation',
                  os.getcwd()]:
    if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, 'checkpoints')):
        project_path = candidate
        break
if project_path is None:
    project_path = os.getcwd()

os.chdir(project_path)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
model_src = os.path.join(project_path, 'Model/Dig-Data_Model-Main')
if model_src not in sys.path:
    sys.path.insert(0, model_src)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Working directory: {os.getcwd()}")
print(f"Runtime device: {device}")

# -- Fixed constants (do not fit these to data; see ADR 0004) --
# Label ordering is the training-time constant from Model/Dig-Data_Model-Main/src/datasets.py::getEmotionDict().
# There is no label-permutation solver in this notebook.
EMO_DICT = {'ang': 0, 'dis': 1, 'fea': 2, 'hap': 3, 'sad': 4, 'sur': 5}
EMOTION_CLASSES = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']
RAW_EMO_KEYS = ['ang', 'dis', 'fea', 'hap', 'sad', 'sur']

# MODEL_ARGS confirmed against the exact training command line recorded in
# Final_result/Origin_training/wrong_stat/RML_{origin,finetune}/*.txt on the handover drive:
#   -lr=5e-5 -ep=40 -mod=tav -bs=4 --img-interval=500 --early-stop=40 --loss=ce
#   --model=mme2e --num-emotions=6 --trans-dim=64 --trans-nlayers=4 --trans-nheads=4
#   --text-lr-factor=10 --text-model-size=large --text-max-len=100
MODEL_ARGS = {
    'num_emotions': 6,
    'modalities': 'tav',
    'feature_dim': 256,
    'trans_nlayers': 4,
    'trans_nheads': 4,
    'trans_dim': 64,
    'text_model_size': 'large',
    'text_max_len': 100,
}

# Canonical data source (ADR 0004). Do not point this at the retired
# RML_RAW_PROCESSED_Face folder -- that folder's meta.pkl was fabricated
# and is not part of the source archive.
DATA_DIR = os.path.join(project_path, 'Model/Dig-Data_Model-Main/data')
MAIN_FOLDER = os.path.join(DATA_DIR, 'BIG_DATA_RAW_PROCESSED_FACE')
META_PATH = os.path.join(MAIN_FOLDER, 'meta_one_hot_label_six_categories.pkl')
SPLIT_DIR = os.path.join(DATA_DIR, 'data_split', 'all_single_label_six_category', 'with_valid')

CHECKPOINTS_DIR = os.path.join(project_path, 'checkpoints')
BASE_CKPT = os.path.join(CHECKPOINTS_DIR, 'base_model.pt')
FT_CKPT = os.path.join(CHECKPOINTS_DIR, 'finetuned_model.pt')

for d in [os.path.join(CHECKPOINTS_DIR, 'activations'),
          os.path.join(project_path, 'results'),
          os.path.join(project_path, 'figures')]:
    os.makedirs(d, exist_ok=True)

def compute_sha256(filepath):
    if not os.path.exists(filepath):
        return "FILE_NOT_FOUND"
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

print("Checkpoint SHA-256:")
print(f"  base_model.pt      {compute_sha256(BASE_CKPT)}")
print(f"  finetuned_model.pt {compute_sha256(FT_CKPT)}")
print("Expected (docs/adr/0004-...):")
print("  base_model.pt      565bc220d187f2286500481fdab4d3b3dc4f92a2006ffb8f02ca4c882bbd82db")
print("  finetuned_model.pt a4c1707f7bcc189d3103b42ea82c9d01f6ec2f992850e8cc1072f38f4dadd0de")
assert compute_sha256(BASE_CKPT) == "565bc220d187f2286500481fdab4d3b3dc4f92a2006ffb8f02ca4c882bbd82db", \
    "base_model.pt does not match the verified checkpoint. Stop and re-sync before continuing."
assert compute_sha256(FT_CKPT) == "a4c1707f7bcc189d3103b42ea82c9d01f6ec2f992850e8cc1072f38f4dadd0de", \
    "finetuned_model.pt does not match the verified checkpoint. Stop and re-sync before continuing."
print("Checkpoint identity verified.")

# Day 0 — Data Loading (Canonical Source) & Halting Fidelity Gate

Loads RML samples from `BIG_DATA_RAW_PROCESSED_FACE` + `meta_one_hot_label_six_categories.pkl`, filtered by the RML split files — mirroring `main.py::get_dataset_iemocap` exactly (same main folder, same meta file, same label dict). Evaluates both checkpoints on the 144-sample test split and **halts** if test accuracy doesn't reproduce the paper's published RML numbers.

In [ ]:
# -- Day 0: Canonical Data Load + Halting Fidelity Gate --
import pickle
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AlbertTokenizer

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

from src.datasets import IEMOCAP, collate_fn
from src.models.e2e import MME2E

assert os.path.exists(META_PATH), (
    f"Canonical meta file not found at {META_PATH}. "
    "Extract BIG_DATA_RAW_PROCESSED_FACE.tar.gz (or its RML-relevant subset) "
    "from the handover drive into Model/Dig-Data_Model-Main/data/ before continuing. "
    "See docs/adr/0004-v3-restart-data-provenance-and-fidelity-gate.md."
)

with open(META_PATH, 'rb') as f:
    meta = pickle.load(f)
print(f"Loaded canonical meta: {len(meta)} total utterances across all six datasets.")

def load_rml_split(phase):
    path = os.path.join(SPLIT_DIR, f'Final_{phase}_split_six_categories_RML.txt')
    ids = open(path).read().splitlines()
    missing = [uid for uid in ids if uid not in meta]
    if missing:
        raise KeyError(
            f"{len(missing)}/{len(ids)} RML {phase} split IDs are not keys in the canonical meta file "
            f"(e.g. {missing[:5]}). The merged corpus may have re-ID'd RML samples during construction; "
            "do not proceed with a partial-ID extraction (see ADR 0004's blocking check)."
        )
    return ids

train_ids = load_rml_split('train')
valid_ids = load_rml_split('valid')
test_ids = load_rml_split('test')
print(f"RML split sizes (all IDs confirmed present in canonical meta): "
      f"train={len(train_ids)} valid={len(valid_ids)} test={len(test_ids)}")
# Fidelity check on split composition itself (Table 4.2 / wrong_stat training log: 518/58/144).
assert (len(train_ids), len(valid_ids), len(test_ids)) == (518, 58, 144), (
    f"Split sizes {(len(train_ids), len(valid_ids), len(test_ids))} don't match the paper's "
    "518/58/144 (Table 4.2). Stop and investigate before continuing."
)

def build_loader(uttr_ids, batch_size=8):
    texts = [meta[uid]['text'] for uid in uttr_ids]
    labels = [meta[uid]['label'] for uid in uttr_ids]  # already one-hot in meta_one_hot_label_six_categories.pkl
    dataset = IEMOCAP(
        main_folder=MAIN_FOLDER,
        utterance_ids=uttr_ids,
        texts=texts,
        labels=labels,
        label_annotations=RAW_EMO_KEYS,
        img_interval=500,
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2,
                       pin_memory=True, collate_fn=collate_fn)

test_loader = build_loader(test_ids)
assert len(test_loader.dataset) == 144, f"Test loader has {len(test_loader.dataset)} samples, expected 144."

tokenizer = AlbertTokenizer.from_pretrained('albert-large-v2')

def evaluate_accuracy(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            _, imgs, img_lens, specs, spec_lens, text_batch, Y = batch
            text_inputs = tokenizer(list(text_batch), return_tensors='pt',
                                     max_length=MODEL_ARGS['text_max_len'],
                                     padding='max_length', truncation=True)
            text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
            imgs = imgs.to(device) if hasattr(imgs, 'to') else torch.tensor(imgs, device=device)
            specs = specs.to(device) if hasattr(specs, 'to') else torch.tensor(specs, device=device)
            logits = model(imgs, img_lens, specs, spec_lens, text_inputs)
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_targets.extend(Y.argmax(-1).cpu().numpy())
    preds, targets = np.array(all_preds), np.array(all_targets)
    return float(np.mean(preds == targets)) * 100.0

def load_model(ckpt_path):
    model = MME2E(args=MODEL_ARGS, device=device).to(device)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict, strict=True)
    return model

print("\nLoading base and fine-tuned models with strict=True (see journal 2026-08-09: "
      "key-diff check confirmed zero missing/mismatched keys for these checkpoints)...")
base_model = load_model(BASE_CKPT)
ft_model = load_model(FT_CKPT)
print("Both checkpoints loaded with strict=True. No missing or unexpected keys.")

print("\nEvaluating unablated test accuracy (fixed getEmotionDict() label order, no permutation search)...")
base_acc = evaluate_accuracy(base_model, test_loader)
ft_acc = evaluate_accuracy(ft_model, test_loader)

TARGET_BASE_ACC = 76.39  # Figure 4.3, For Feature Analysis.pdf; cross-checked against
TARGET_FT_ACC = 79.86    # Final_result/Origin_training/wrong_stat/RML_*/*.txt "Best performance" test row.
TOLERANCE = 1.0

print(f"\n=== Day 0 Fidelity Gate ===")
print(f"Base model test accuracy:       {base_acc:.2f}%  (target {TARGET_BASE_ACC}% +/- {TOLERANCE}%)")
print(f"Fine-tuned model test accuracy: {ft_acc:.2f}%  (target {TARGET_FT_ACC}% +/- {TOLERANCE}%)")

assert abs(base_acc - TARGET_BASE_ACC) < TOLERANCE, (
    f"HALT: base model test accuracy {base_acc:.2f}% does not reproduce the paper's {TARGET_BASE_ACC}%. "
    "Do not proceed to Week 1. Re-check data provenance (ADR 0004) before re-running this cell."
)
assert abs(ft_acc - TARGET_FT_ACC) < TOLERANCE, (
    f"HALT: fine-tuned model test accuracy {ft_acc:.2f}% does not reproduce the paper's {TARGET_FT_ACC}%. "
    "Do not proceed to Week 1. Re-check data provenance (ADR 0004) before re-running this cell."
)
print("\nFidelity gate PASSED. Safe to proceed to the falsification pair and Week 1.")

# Day 0 (cont.) — Falsification Pair

Before any real ablation sweep: (a) ablate all 64 audio dimensions (full-modality knockout) — accuracy must move substantially; (b) ablate 5 random dimensions — the drop must be near zero. Run on the base model only; this is a harness sanity check, not a scientific result. If this cell fails, the harness is still broken and Week 2 must not run.

In [ ]:
# -- Day 0: Falsification Pair (harness sanity check, not a scientific result) --

class MeanAblationHook:
    """Overwrites `target_indices` of the hooked module's output with `mean_vector`.

    Registered on model.a_transformer, whose forward (called with get_cls=True)
    returns the 2D [B, 64] CLS-token representation that feeds directly into
    a_out and the fusion sum (confirmed in journal 2026-08-09 / ADR 0004 --
    there is no separate FFN layer to distinguish this from).
    """
    def __init__(self, target_indices, mean_vector):
        self.target_indices = target_indices
        self.mean_vector = mean_vector

    def __call__(self, module, input, output):
        out = output[0] if isinstance(output, tuple) else output
        modified = out.clone()
        clamp = self.mean_vector.to(modified.device)
        modified[:, self.target_indices] = clamp[self.target_indices]
        if isinstance(output, tuple):
            return (modified,) + output[1:]
        return modified

def capture_a_transformer_mean(model, loader):
    """Mean per-dimension activation of model.a_transformer's output over `loader`.
    Used only for this sanity check; Week 2's real ablation mean is computed
    over the training split, not the test split (see Day 6-7 cell)."""
    acts = []
    def hook_fn(module, input, output):
        out = output[0] if isinstance(output, tuple) else output
        acts.append(out.detach().cpu())
    handle = model.a_transformer.register_forward_hook(hook_fn)
    evaluate_accuracy(model, loader)  # discard accuracy, just want the hook to fire
    handle.remove()
    return torch.cat(acts, dim=0).mean(dim=0)

def evaluate_with_ablation(model, loader, target_indices, mean_vector):
    hook = MeanAblationHook(target_indices, mean_vector)
    handle = model.a_transformer.register_forward_hook(hook)
    acc = evaluate_accuracy(model, loader)
    handle.remove()
    return acc

print("Computing test-set mean activation of model.a_transformer for the falsification pair...")
falsification_mean = capture_a_transformer_mean(base_model, test_loader)

full_knockout_acc = evaluate_with_ablation(base_model, test_loader, list(range(64)), falsification_mean)

rng = np.random.default_rng(0)
random_5 = rng.choice(64, size=5, replace=False).tolist()
random_5_acc = evaluate_with_ablation(base_model, test_loader, random_5, falsification_mean)

full_knockout_drop = base_acc - full_knockout_acc
random_5_drop = base_acc - random_5_acc

print(f"\n=== Falsification Pair (base model, unablated = {base_acc:.2f}%) ===")
print(f"Full 64-dim audio knockout: {full_knockout_acc:.2f}%  (drop = {full_knockout_drop:+.2f} pts)")
print(f"Random 5-dim control:       {random_5_acc:.2f}%  (drop = {random_5_drop:+.2f} pts, indices={random_5})")

assert full_knockout_drop > 10.0, (
    f"HALT: full audio knockout only moved accuracy by {full_knockout_drop:.2f} points. "
    "The ablation hook is not affecting the model's predictions the way it should -- "
    "do not trust any k-sweep result until this is understood."
)
assert random_5_drop < 10.0, (
    f"HALT: a random 5-neuron ablation moved accuracy by {random_5_drop:.2f} points, "
    "comparable to a full 64-dim knockout. This means ablating almost any 5 neurons "
    "produces a large effect, which would make any single class's top-5 result "
    "uninterpretable as class-selective. Investigate before proceeding."
)
print("\nFalsification pair PASSED: the hook has a real, appropriately-scaled causal effect.")

# Week 1 — Days 1-2: Extract and Cache Activations

Forward pass over the full RML dataset (train + valid + test, per protocol Day 1-2: "you're not training anything, just observing") for both models, caching `model.a_transformer`'s output (the 64-d CLS representation) and the ground-truth label for every sample. Probe fitting later uses only the train split; the test split stays held out throughout.

In [ ]:
# -- Week 1, Days 1-2: Extract and cache activations (train=518, valid=58, test=144) --

ACTS_DIR = os.path.join(CHECKPOINTS_DIR, 'activations')

def extract_activations(model, loader):
    model.eval()
    acts, labels = [], []
    def hook_fn(module, input, output):
        out = output[0] if isinstance(output, tuple) else output
        acts.append(out.detach().cpu())
    handle = model.a_transformer.register_forward_hook(hook_fn)
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting activations", leave=False):
            _, imgs, img_lens, specs, spec_lens, text_batch, Y = batch
            text_inputs = tokenizer(list(text_batch), return_tensors='pt',
                                     max_length=MODEL_ARGS['text_max_len'],
                                     padding='max_length', truncation=True)
            text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
            imgs = imgs.to(device) if hasattr(imgs, 'to') else torch.tensor(imgs, device=device)
            specs = specs.to(device) if hasattr(specs, 'to') else torch.tensor(specs, device=device)
            _ = model(imgs, img_lens, specs, spec_lens, text_inputs)
            labels.extend(Y.argmax(-1).cpu().numpy())
    handle.remove()
    return torch.cat(acts, dim=0), torch.tensor(labels, dtype=torch.long)

train_loader = build_loader(train_ids)
valid_loader = build_loader(valid_ids)
# test_loader already built during the Day 0 fidelity gate.

cache_spec = [
    ('base', base_model, {'train': train_loader, 'valid': valid_loader, 'test': test_loader}),
    ('finetuned', ft_model, {'train': train_loader, 'valid': valid_loader, 'test': test_loader}),
]

activations = {}  # activations[model_name][split] = (acts, labels)
for model_name, model, loaders in cache_spec:
    activations[model_name] = {}
    for split_name, loader in loaders.items():
        acts_path = os.path.join(ACTS_DIR, f'{model_name}_{split_name}_acts.pt')
        labels_path = os.path.join(ACTS_DIR, f'{model_name}_{split_name}_labels.pt')
        if os.path.exists(acts_path) and os.path.exists(labels_path):
            acts, labels = torch.load(acts_path), torch.load(labels_path)
            print(f"Resume gate: loaded cached {model_name}/{split_name} ({acts.shape[0]} samples).")
        else:
            print(f"Extracting {model_name}/{split_name}...")
            acts, labels = extract_activations(model, loader)
            torch.save(acts, acts_path)
            torch.save(labels, labels_path)
        activations[model_name][split_name] = (acts, labels)

# Cross-model label consistency check: both models must see the same samples in the same order.
assert torch.equal(activations['base']['train'][1], activations['finetuned']['train'][1])
assert torch.equal(activations['base']['test'][1], activations['finetuned']['test'][1])
for model_name in activations:
    for split_name, (acts, labels) in activations[model_name].items():
        expected_n = {'train': 518, 'valid': 58, 'test': 144}[split_name]
        assert acts.shape == (expected_n, 64), f"{model_name}/{split_name}: expected ({expected_n}, 64), got {tuple(acts.shape)}"

print("\nWeek 1 Days 1-2 complete. Cached shapes:")
for model_name in activations:
    for split_name, (acts, labels) in activations[model_name].items():
        print(f"  {model_name:10s} {split_name:6s} acts={tuple(acts.shape)} labels={tuple(labels.shape)}")

# Week 1 — Day 3: Fit Sparse L1-Penalized Probes (train split only)

One binary L1-logistic probe per (model, class), fit strictly on the 518-sample train split. Ranks the 64 dimensions by absolute weight magnitude, giving a per-class, per-model candidate neuron ranking. Never fit on valid or test.

In [ ]:
# -- Week 1, Day 3: Fit L1 probes on the train split, rank neurons per class --
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def fit_class_probes(train_acts, train_labels):
    """Returns {class_name: (scaler, probe, ranked_indices, ranked_weights)}."""
    X_train = train_acts.numpy()
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    probes = {}
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        y_bin = (train_labels.numpy() == class_idx).astype(int)
        probe = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, max_iter=2000)
        probe.fit(X_train_scaled, y_bin)
        weights = probe.coef_[0]
        ranked_indices = np.argsort(-np.abs(weights))
        probes[class_name] = {
            'scaler': scaler,
            'probe': probe,
            'ranked_indices': ranked_indices.tolist(),
            'ranked_weights': weights[ranked_indices].tolist(),
        }
    return probes

class_probes = {}
for model_name in ['base', 'finetuned']:
    train_acts, train_labels = activations[model_name]['train']
    class_probes[model_name] = fit_class_probes(train_acts, train_labels)
    print(f"\n=== {model_name.title()} model: top-5 neurons per class (train-fit) ===")
    for class_name in EMOTION_CLASSES:
        info = class_probes[model_name][class_name]
        top5 = [(idx, round(w, 3)) for idx, w in zip(info['ranked_indices'][:5], info['ranked_weights'][:5])]
        print(f"  {class_name:10s} {top5}")

# Week 1 — Day 4: Sanity-Check Probes (Held-Out Test AUC)

ROC-AUC on the 144-sample test split, held out from Day 3's fit. Protocol fallback: if mean AUC < 0.65 across the six classes, or more than 2 classes have AUC < 0.55, the layer doesn't carry a clean linear signal and Day 0's layer choice should be revisited.

In [ ]:
# -- Week 1, Day 4: Held-out test AUC sanity check --

probe_auc = {}
for model_name in ['base', 'finetuned']:
    test_acts, test_labels = activations[model_name]['test']
    X_test = test_acts.numpy()
    y_test = test_labels.numpy()
    print(f"\n=== {model_name.title()} model: held-out test AUC (N_test={len(y_test)}) ===")
    aucs = []
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        info = class_probes[model_name][class_name]
        X_test_scaled = info['scaler'].transform(X_test)
        y_bin = (y_test == class_idx).astype(int)
        scores = info['probe'].predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_bin, scores)
        aucs.append(auc)
        print(f"  {class_name:10s} AUC = {auc:.4f}")
    mean_auc = float(np.mean(aucs))
    n_below_055 = sum(a < 0.55 for a in aucs)
    print(f"  Mean AUC = {mean_auc:.4f}  |  classes with AUC < 0.55: {n_below_055}")
    probe_auc[model_name] = {'per_class': dict(zip(EMOTION_CLASSES, aucs)), 'mean': mean_auc}
    if mean_auc < 0.65 or n_below_055 > 2:
        print(f"  WARNING: {model_name} model fails the Day 4 fallback threshold "
              "(mean AUC < 0.65 or >2 classes < 0.55). Per protocol Day 4, revisit the "
              "Day 0 layer choice (try the CLS-token layer of a different modality) "
              "before continuing to Week 2.")

with open(os.path.join(project_path, 'results', 'week1_probe_auc.json'), 'w') as f:
    import json
    json.dump(probe_auc, f, indent=2)
print("\nSaved results/week1_probe_auc.json")

# Week 2 — Days 6-10: Causal Ablation Sweep

Train-set mean ablation (computed once, over the 518-sample train split, matching protocol Day 6-7). For each model and class, ablate the top-k neurons (k = 1, 3, 5, 10, per protocol; 16/32/48/64 added for the dose-response curve) and evaluate on the **144-sample test split** — the same split the Day 0 fidelity gate and falsification pair used. A class-selectivity ratio of target-class drop over mean non-target drop >= 2.5x (ADR 0001) identifies a causally class-selective feature set; k=5 is the primary table entry.

In [ ]:
# -- Week 2, Days 6-7: Train-set mean ablation vector (MeanAblationHook defined in the Day 0 falsification cell) --

train_mean = {}
for model_name in ['base', 'finetuned']:
    train_acts, _ = activations[model_name]['train']
    train_mean[model_name] = train_acts.mean(dim=0)
    print(f"{model_name.title()} train-set mean vector: shape={tuple(train_mean[model_name].shape)}, "
          f"mean={train_mean[model_name].mean().item():.4f}, std={train_mean[model_name].std().item():.4f}")

def per_class_accuracies(preds, targets):
    accs = {'overall': float(np.mean(preds == targets)) * 100.0}
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        mask = targets == class_idx
        accs[class_name] = float(np.mean(preds[mask] == targets[mask])) * 100.0 if mask.sum() > 0 else float('nan')
    return accs

def evaluate_per_class(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            _, imgs, img_lens, specs, spec_lens, text_batch, Y = batch
            text_inputs = tokenizer(list(text_batch), return_tensors='pt',
                                     max_length=MODEL_ARGS['text_max_len'],
                                     padding='max_length', truncation=True)
            text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
            imgs = imgs.to(device) if hasattr(imgs, 'to') else torch.tensor(imgs, device=device)
            specs = specs.to(device) if hasattr(specs, 'to') else torch.tensor(specs, device=device)
            logits = model(imgs, img_lens, specs, spec_lens, text_inputs)
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_targets.extend(Y.argmax(-1).cpu().numpy())
    return per_class_accuracies(np.array(all_preds), np.array(all_targets))

K_VALUES = [1, 3, 5, 10, 16, 32, 48, 64]

def run_ablation_sweep(model, model_name, loader):
    baseline = evaluate_per_class(model, loader)
    print(f"{model_name.title()} unablated baseline: overall={baseline['overall']:.2f}%")
    sweep = {'baseline': baseline}
    for class_name in tqdm(EMOTION_CLASSES, desc=f"{model_name} sweep"):
        ranked = class_probes[model_name][class_name]['ranked_indices']
        class_results = {}
        for k in K_VALUES:
            target_idx = ranked[:k]
            hook = MeanAblationHook(target_idx, train_mean[model_name])
            handle = model.a_transformer.register_forward_hook(hook)
            ablated = evaluate_per_class(model, loader)
            handle.remove()
            target_drop = baseline[class_name] - ablated[class_name]
            non_target_drops = [baseline[c] - ablated[c] for c in EMOTION_CLASSES if c != class_name]
            mean_non_target_drop = float(np.mean(non_target_drops))
            # Selectivity ratio uses max(non_target_drop, small_eps) in the denominator only to avoid
            # division by ~0; the raw non_target_drop is always reported alongside it.
            denom = max(mean_non_target_drop, 1e-6)
            selectivity_ratio = target_drop / denom
            class_results[k] = {
                'ablated_accs': ablated,
                'target_drop': target_drop,
                'mean_non_target_drop': mean_non_target_drop,
                'selectivity_ratio': selectivity_ratio,
            }
        sweep[class_name] = class_results
        r5 = class_results[5]
        print(f"  [{class_name:10s}] k=5 target_drop={r5['target_drop']:+.2f}pp  "
              f"non_target_drop={r5['mean_non_target_drop']:+.2f}pp  "
              f"selectivity={r5['selectivity_ratio']:.2f}x")
    return sweep

ablation_sweeps = {}
for model_name, model in [('base', base_model), ('finetuned', ft_model)]:
    ablation_sweeps[model_name] = run_ablation_sweep(model, model_name, test_loader)

for model_name in ablation_sweeps:
    path = os.path.join(project_path, 'results', f'week2_{model_name}_ablation_sweep.json')
    with open(path, 'w') as f:
        import json
        json.dump(ablation_sweeps[model_name], f, indent=2, default=float)
    print(f"Saved {path}")

# Week 3 — Days 11-15: Does Fine-Tuning Preserve or Reassign the Neurons?

Per-class selectivity vectors (`mean(activation | class c) - mean(activation | class != c)`, computed on the train split for both models), cosine similarity between base and fine-tuned selectivity vectors, and the decisive test: ablate the **base model's** top-5 neurons for each class directly inside the **fine-tuned model** and measure the drop. Per protocol Day 14, this ablation-transfer result is the headline test if it disagrees with the cosine-similarity number.

In [ ]:
# -- Week 3, Days 11-13: Selectivity vectors + cosine similarity (train split) --

def selectivity_vectors(train_acts, train_labels):
    X = train_acts.numpy()
    y = train_labels.numpy()
    vectors = {}
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        in_class = X[y == class_idx].mean(axis=0)
        out_class = X[y != class_idx].mean(axis=0)
        vectors[class_name] = in_class - out_class
    return vectors

selectivity = {}
for model_name in ['base', 'finetuned']:
    train_acts, train_labels = activations[model_name]['train']
    selectivity[model_name] = selectivity_vectors(train_acts, train_labels)

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

cosine_similarities = {
    class_name: cosine_sim(selectivity['base'][class_name], selectivity['finetuned'][class_name])
    for class_name in EMOTION_CLASSES
}
print("=== Day 13: Base vs. Fine-Tuned selectivity vector cosine similarity ===")
for class_name, sim in cosine_similarities.items():
    print(f"  {class_name:10s} cos_sim = {sim:+.4f}")

In [ ]:
# -- Week 3, Day 14: Ablation transfer -- base model's top-5 neurons ablated INSIDE the fine-tuned model --

ft_baseline = ablation_sweeps['finetuned']['baseline']
transfer_results = {}
for class_name in EMOTION_CLASSES:
    base_top5 = class_probes['base'][class_name]['ranked_indices'][:5]
    hook = MeanAblationHook(base_top5, train_mean['finetuned'])
    handle = ft_model.a_transformer.register_forward_hook(hook)
    ablated = evaluate_per_class(ft_model, test_loader)
    handle.remove()
    ft_drop_using_base_neurons = ft_baseline[class_name] - ablated[class_name]
    transfer_results[class_name] = {
        'base_top5_indices': base_top5,
        'ft_drop_using_base_neurons': ft_drop_using_base_neurons,
        'ft_ablated_accs': ablated,
    }
    print(f"  {class_name:10s} base-top5={base_top5}  "
          f"FT drop when ablated with base's neurons: {ft_drop_using_base_neurons:+.2f}pp")

with open(os.path.join(project_path, 'results', 'week3_ablation_transfer.json'), 'w') as f:
    import json
    json.dump(transfer_results, f, indent=2, default=float)
print("Saved results/week3_ablation_transfer.json")

# Week 3 — Day 15: Compile Transfer Retention Table

Retention Ratio `R = ft_drop / base_drop` (both drops taken from the same k=5 ablation, `ft_drop` using the base model's neuron indices per Day 14). Per ADR 0004, the previous ε=0.05 "N/A" screen is **withdrawn** — it was written to hide artifacts produced by the v1/v2 broken harness, not a genuine statistical need. This cell reports the raw ratio for every class and flags (does not suppress) cases where `base_drop` is small enough that R is noisy.

In [ ]:
# -- Week 3, Day 15: Transfer Retention Ratio table --

FLAG_THRESHOLD = 5.0  # pp; below this, note that R may be noisy (informational only, per ADR 0004)

retention_table = []
for class_name in EMOTION_CLASSES:
    base_drop = ablation_sweeps['base'][class_name][5]['target_drop']
    ft_drop = transfer_results[class_name]['ft_drop_using_base_neurons']
    r = ft_drop / base_drop if abs(base_drop) > 1e-9 else float('nan')
    if r >= 0.80:
        outcome = 'Substrate Preservation'
    elif r >= 0.20:
        outcome = 'Substrate Reassignment'
    elif not np.isnan(r):
        outcome = 'Substrate Dispersion'
    else:
        outcome = 'Undefined (base_drop == 0)'
    flag = base_drop < FLAG_THRESHOLD
    retention_table.append({
        'class': class_name,
        'base_drop_pp': base_drop,
        'ft_drop_pp': ft_drop,
        'R': r,
        'outcome': outcome,
        'small_base_drop_flag': flag,
        'cosine_similarity': cosine_similarities[class_name],
    })

import pandas as pd
retention_df = pd.DataFrame(retention_table)
print(retention_df.to_string(index=False))
retention_df.to_csv(os.path.join(project_path, 'results', 'week3_transfer_retention.csv'), index=False)
print("\nSaved results/week3_transfer_retention.csv")

flagged = retention_df[retention_df['small_base_drop_flag']]
if len(flagged) > 0:
    print(f"\nNote: {len(flagged)} class(es) have base_drop < {FLAG_THRESHOLD}pp, "
          "so R for those classes is sensitive to small denominator noise. Reported, not suppressed.")

# Week 4 — Days 16-18: Publication Tables & Dose-Response Figure

Table VII (base model, k=5 target vs. mean non-target drop per class), Table VIII (fine-tuned model, same), Table IX (cosine similarity + ablation-transfer drop per class, matching the columns in the IEEE paper's Section VI-D placeholder). Dose-response figure across k=1..64 for both models.

In [ ]:
# -- Week 4, Days 16-18: Export Tables VII/VIII/IX and dose-response figure --
import matplotlib.pyplot as plt

def build_table_vii_viii(model_name):
    rows = []
    for class_name in EMOTION_CLASSES:
        r = ablation_sweeps[model_name][class_name][5]
        rows.append({
            'class': class_name,
            'target_class_drop_pp': r['target_drop'],
            'mean_non_target_drop_pp': r['mean_non_target_drop'],
            'selectivity_ratio': r['selectivity_ratio'],
        })
    return pd.DataFrame(rows)

table_vii = build_table_vii_viii('base')       # Table VII: base model, k=5
table_viii = build_table_vii_viii('finetuned') # Table VIII: fine-tuned model, k=5
table_ix = retention_df[['class', 'cosine_similarity', 'ft_drop_pp']].rename(
    columns={'ft_drop_pp': 'delta_acc_target_class_base_neurons_in_ft'})

table_vii.to_csv(os.path.join(project_path, 'results', 'table_vii_base_ablation_k5.csv'), index=False)
table_viii.to_csv(os.path.join(project_path, 'results', 'table_viii_finetuned_ablation_k5.csv'), index=False)
table_ix.to_csv(os.path.join(project_path, 'results', 'table_ix_cosine_and_transfer.csv'), index=False)

print("=== Table VII (base model, k=5) ===")
print(table_vii.to_string(index=False))
print("\n=== Table VIII (fine-tuned model, k=5) ===")
print(table_viii.to_string(index=False))
print("\n=== Table IX (cosine similarity + ablation transfer) ===")
print(table_ix.to_string(index=False))

# Dose-response figure: target-class drop vs k, one subplot per model.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, model_name in zip(axes, ['base', 'finetuned']):
    for class_name in EMOTION_CLASSES:
        drops = [ablation_sweeps[model_name][class_name][k]['target_drop'] for k in K_VALUES]
        ax.plot(K_VALUES, drops, marker='o', label=class_name)
    ax.set_title(f'{model_name.title()} model')
    ax.set_xlabel('k (top-k neurons ablated)')
    ax.axhline(0, color='gray', linewidth=0.5)
axes[0].set_ylabel('Target-class accuracy drop (pp)')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
fig.suptitle('Dose-response: target-class accuracy drop vs. k')
fig.tight_layout()
fig_path = os.path.join(project_path, 'figures', 'dose_response_k_sweep.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"\nSaved {fig_path}")
plt.show()

print("\nWeek 4 Days 16-18 complete. All Table VII/VIII/IX CSVs and the dose-response figure are in results/ and figures/.")

# Handoff

At this point `results/` contains: `week1_probe_auc.json`, `week2_{base,finetuned}_ablation_sweep.json`, `week3_ablation_transfer.json`, `week3_transfer_retention.csv`, `table_{vii,viii,ix}_*.csv`; `figures/dose_response_k_sweep.png`; `checkpoints/activations/*.pt` (resumable — Days 1-2's resume gate skips re-extraction if these already exist).

**Next steps (Week 4, Days 19-21+, manual):**
1. Write a journal entry for the session that ran this notebook, following the 7-part schema in `journals/GUIDELINES.md` — record the actual fidelity-gate numbers, probe AUCs, and whether the falsification pair passed.
2. Draft Section VI-D's `[SUMMARIZE HERE ONCE RESULTS ARE AVAILABLE]` placeholders in the IEEE paper docx using Table VII/VIII/IX's real numbers — state plainly whether drops were class-selective (>= 2.5x, ADR 0001) and whether fine-tuning preserved or reassigned the neurons per class.
3. State the Audio/Text near-tie explicitly in the write-up (ADR 0004 decision 5) — do not present Audio as an unambiguous dominant modality.
4. If Day 4's probe AUC fails the fallback threshold, or the Day 0 fidelity gate fails, stop and fix that before trusting anything downstream — do not let a warning become a reason to continue (this is exactly how v1 and v2 failed).